In [1]:
from pathlib import Path
import pandas as pd

In [2]:

# 50 states + D.C. mapped to (State Name, 2-character string FIPS)
STATES = {
    "AL": ("Alabama", "01"),
    "AK": ("Alaska", "02"),
    "AZ": ("Arizona", "04"),
    "AR": ("Arkansas", "05"),
    "CA": ("California", "06"),
    "CO": ("Colorado", "08"),
    "CT": ("Connecticut", "09"),
    "DE": ("Delaware", "10"),
    "DC": ("District of Columbia", "11"),
    "FL": ("Florida", "12"),
    "GA": ("Georgia", "13"),
    "HI": ("Hawaii", "15"),
    "ID": ("Idaho", "16"),
    "IL": ("Illinois", "17"),
    "IN": ("Indiana", "18"),
    "IA": ("Iowa", "19"),
    "KS": ("Kansas", "20"),
    "KY": ("Kentucky", "21"),
    "LA": ("Louisiana", "22"),
    "ME": ("Maine", "23"),
    "MD": ("Maryland", "24"),
    "MA": ("Massachusetts", "25"),
    "MI": ("Michigan", "26"),
    "MN": ("Minnesota", "27"),
    "MS": ("Mississippi", "28"),
    "MO": ("Missouri", "29"),
    "MT": ("Montana", "30"),
    "NE": ("Nebraska", "31"),
    "NV": ("Nevada", "32"),
    "NH": ("New Hampshire", "33"),
    "NJ": ("New Jersey", "34"),
    "NM": ("New Mexico", "35"),
    "NY": ("New York", "36"),
    "NC": ("North Carolina", "37"),
    "ND": ("North Dakota", "38"),
    "OH": ("Ohio", "39"),
    "OK": ("Oklahoma", "40"),
    "OR": ("Oregon", "41"),
    "PA": ("Pennsylvania", "42"),
    "RI": ("Rhode Island", "44"),
    "SC": ("South Carolina", "45"),
    "SD": ("South Dakota", "46"),
    "TN": ("Tennessee", "47"),
    "TX": ("Texas", "48"),
    "UT": ("Utah", "49"),
    "VT": ("Vermont", "50"),
    "VA": ("Virginia", "51"),
    "WA": ("Washington", "53"),
    "WV": ("West Virginia", "54"),
    "WI": ("Wisconsin", "55"),
    "WY": ("Wyoming", "56"),
}



In [3]:

frames = []

for abbr, (name, fips) in STATES.items():
  series_id = f"{abbr}UR"  # Monthly, Seasonally Adjusted Unemployment Rate
  url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd=2015-01-01&coed=2024-12-31"

  df = pd.read_csv(url)
  df.columns = ["date", "unemployment_rate"]

  df["date"] = pd.to_datetime(df["date"])
  df["year"] = df["date"].dt.year
  df["unemployment_rate"] = pd.to_numeric(
      df["unemployment_rate"], errors="coerce"
  )

  # Explicitly store FIPS as a 2-character string
  df["state_fips"] = str(fips).zfill(2)
  df["state_abbr"] = abbr
  df["state_name"] = name

  frames.append(df)


In [4]:

# Combine all monthly data (2015-2024)
monthly_df = pd.concat(frames, ignore_index=True)
monthly_df = monthly_df[
    ["state_fips", "state_abbr", "state_name", "date", "year", "unemployment_rate"]
]

In [5]:

# Aggregate to annual mean by state and year for dataset integration
annual_df = (
    monthly_df.groupby(
        ["state_fips", "state_abbr", "state_name", "year"], as_index=False
    )["unemployment_rate"]
    .mean()
    .round(2)
)


In [6]:

# Save to your Unemployment2015-2024 folder
out_dir = Path("Unemployment2015-2024")
out_dir.mkdir(exist_ok=True)

monthly_df.to_csv(out_dir / "state_unemployment_monthly_2015_2024.csv", index=False)
annual_df.to_csv(out_dir / "state_unemployment_annual_2015_2024.csv", index=False)

print(
    f"Saved {len(monthly_df)} monthly rows and {len(annual_df)} annual rows."
)

Saved 6120 monthly rows and 510 annual rows.
